# Priorização de clientes para uma campanha bancária

**Pergunta:** com informações disponíveis antes de uma ligação, é possível ordenar clientes pela chance de contratar um depósito a prazo?

Este notebook usa os 41.188 registros ordenados por data de `bank-additional-full.csv`, do conjunto [Bank Marketing (UCI)](https://archive.ics.uci.edu/dataset/222/bank+marketing). Ele baixa o arquivo da fonte original durante a execução. É preciso ter acesso à internet no Colab. Os resultados relatados no texto vêm de uma execução anterior; execute todas as células para reproduzi-los.


## 1. Leitura e exploração dos dados

In [ ]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

import pandas as pd

url = "https://archive.ics.uci.edu/static/public/222/bank%2Bmarketing.zip"
with urlopen(url, timeout=60) as resposta:
    pacote = resposta.read()

with ZipFile(BytesIO(pacote)) as arquivo_externo:
    nome_interno = next(
        nome for nome in arquivo_externo.namelist()
        if nome.endswith("bank-additional.zip")
    )
    pacote_adicional = arquivo_externo.read(nome_interno)

with ZipFile(BytesIO(pacote_adicional)) as arquivo_dados:
    nome_csv = next(
        nome for nome in arquivo_dados.namelist()
        if nome.endswith("bank-additional-full.csv")
    )
    with arquivo_dados.open(nome_csv) as csv:
        df = pd.read_csv(csv, sep=";")

assert df.shape == (41188, 21), f"Dimensão inesperada: {df.shape}"
print(df.shape)
display(df.head())


In [ ]:
display(df["y"].value_counts(normalize=True))
display(df.isna().sum().sort_values(ascending=False).head())

A classe positiva representa 11,3% das observações; portanto, o desempenho dos modelos será comparado com uma referência simples, e não avaliado apenas por acurácia.

In [ ]:
display(df.isna().sum())

In [ ]:
display(df.eq("unknown").sum().sort_values(ascending=False))

In [ ]:
with ZipFile(BytesIO(pacote_adicional)) as arquivo_dados:
    nome_dicionario = next(
        nome for nome in arquivo_dados.namelist()
        if nome.endswith("bank-additional-names.txt")
    )
    texto = arquivo_dados.read(nome_dicionario).decode("utf-8", errors="replace")

for linha in texto.splitlines():
    if "pdays" in linha.lower():
        print(linha)

display(df["pdays"].value_counts().head(10))


### O código especial de `pdays`

O dicionário de dados descreve `pdays = 999` como ausência de contato anterior. Entretanto, a comparação com `previous` e `poutcome` revelou **4.110 registros** com `pdays = 999`, `previous > 0` e `poutcome = failure`. Portanto, `999` **não identifica de forma confiável todos os clientes sem histórico anterior** nesta base; o motivo da divergência não pode ser determinado só pelos dados.

Usamos `previous > 0` para verificar a existência de contato anterior e `pdays != 999` apenas para indicar que há um intervalo de dias utilizável. Não interpretamos `999` como 999 dias.

In [ ]:
display(pd.crosstab(df["poutcome"], df["previous"].gt(0)))
display(pd.crosstab(df["poutcome"], df["pdays"].ne(999)))

df["pdays_informado"] = df["pdays"].ne(999).astype(int)

As categorias `unknown` são mantidas como categorias próprias nesta primeira versão. `pdays_informado` descreve apenas a disponibilidade de um intervalo; `previous` e `poutcome` preservam o histórico anterior.

## 2. Divisão cronológica e mudança entre períodos

In [ ]:
colunas = [
    "age", "job", "marital", "education", "default",
    "housing", "loan", "previous", "poutcome",
    "pdays_informado"
]

X = df[colunas]
y = df["y"].eq("yes").astype(int)

corte = int(len(df) * 0.8)

X_treino, X_teste = X.iloc[:corte], X.iloc[corte:]
y_treino, y_teste = y.iloc[:corte], y.iloc[corte:]

print("Treino:", X_treino.shape, "Taxa de contratação:", y_treino.mean().round(3))
print("Teste: ", X_teste.shape, "Taxa de contratação:", y_teste.mean().round(3))

In [ ]:
df_analise = df.copy()
df_analise["periodo"] = ["treino"] * corte + ["teste"] * (len(df) - corte)

display(
    pd.crosstab(
        df_analise["periodo"],
        df_analise["poutcome"],
        normalize="index"
    ).round(3)
)

display(
    df_analise.groupby(["periodo", "poutcome"])["y"]
    .apply(lambda valores: valores.eq("yes").mean())
    .unstack()
    .round(3)
)

### Mudança entre os períodos

A divisão inicial por ordem temporal revelou uma diferença expressiva na taxa de contratação: **6,4% nos primeiros 80% dos registros** e **30,8% nos 20% finais**. Também mudou o perfil do histórico das campanhas: a participação de clientes com sucesso anterior passou de **0,5% para 14,6%**. Além disso, a taxa de contratação aumentou mesmo dentro dos grupos de `poutcome`.

Portanto, a diferença não se explica apenas pela maior presença de clientes com sucesso anterior. Os dados sugerem uma mudança temporal na composição dos registros e na relação entre as variáveis e a contratação; ainda não identificamos sua causa.

Para avaliar o modelo de forma mais organizada, dividiremos os dados cronologicamente em **treino (60%)**, **validação (20%)** e **teste (20%)**. A validação servirá para comparar modelos e escolhas de processamento. Já examinamos a taxa e a composição dos registros finais nesta exploração, portanto o teste não é totalmente cego. Não usaremos suas métricas para escolher modelos ou parâmetros.


In [ ]:
n = len(df)
fim_treino = int(n * 0.60)
fim_validacao = int(n * 0.80)

X_treino = X.iloc[:fim_treino]
X_validacao = X.iloc[fim_treino:fim_validacao]
X_teste = X.iloc[fim_validacao:]

y_treino = y.iloc[:fim_treino]
y_validacao = y.iloc[fim_treino:fim_validacao]
y_teste = y.iloc[fim_validacao:]

for nome, alvo in [
    ("Treino", y_treino),
    ("Validação", y_validacao),
    ("Teste", y_teste),
]:
    print(nome, len(alvo), f"contratação: {alvo.mean():.1%}")

A taxa de contratação cresce ao longo das três partes cronológicas (4,8%, 11,1% e 30,8%). Isso indica mudança de distribuição. A taxa e a composição do teste já foram examinadas na exploração; a comparação e a escolha dos modelos serão feitas somente na validação.

A variável `duration` foi excluída porque a duração da ligação só é conhecida depois do contato. `campaign` também ficou fora do primeiro modelo, pois sua definição inclui o contato atual. As variáveis são selecionadas antes de ajustar qualquer transformação; os transformadores do `Pipeline` aprendem apenas no treino.

## 3. Primeiros modelos e avaliação na validação

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

colunas_categoricas = X_treino.select_dtypes(include="object").columns.tolist()
colunas_numericas = X_treino.select_dtypes(exclude="object").columns.tolist()

preprocessamento = ColumnTransformer(
    transformers=[
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), colunas_categoricas),
        ("numericas", StandardScaler(), colunas_numericas),
    ]
)

modelo_logistico = Pipeline([
    ("preprocessamento", preprocessamento),
    ("modelo", LogisticRegression(max_iter=1000)),
])

modelo_logistico.fit(X_treino, y_treino)

In [ ]:
import numpy as np
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

prob_validacao = modelo_logistico.predict_proba(X_validacao)[:, 1]
prev_validacao = (prob_validacao >= 0.5).astype(int)

print(f"Taxa de contratação na validação: {y_validacao.mean():.3f}")
print(f"Average precision: {average_precision_score(y_validacao, prob_validacao):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_validacao, prob_validacao):.3f}")
print(f"Precision com corte de 0,5: {precision_score(y_validacao, prev_validacao, zero_division=0):.3f}")
print(f"Recall com corte de 0,5: {recall_score(y_validacao, prev_validacao, zero_division=0):.3f}")

quantidade_top = int(np.ceil(len(y_validacao) * 0.10))
indices_top = np.argsort(prob_validacao)[-quantidade_top:]

taxa_top_10 = y_validacao.iloc[indices_top].mean()
taxa_geral = y_validacao.mean()

print(f"Clientes selecionados no top 10%: {quantidade_top}")
print(f"Contratação no top 10%: {taxa_top_10:.3f}")
print(f"Lift do top 10%: {taxa_top_10 / taxa_geral:.2f}x")

### Primeira avaliação da regressão logística

O modelo foi treinado nos primeiros 60% dos registros e avaliado nos 20% seguintes, sem usar o conjunto de teste nesta etapa. Na validação, a taxa geral de contratação é **11,1%**.

O modelo obteve **ROC-AUC de 0,613** e **average precision de 0,183**, indicando alguma capacidade de ordenar os clientes por chance de contratação, ainda que limitada. Entre os **10% com maior probabilidade prevista**, a taxa observada de contratação foi **23,1%**, ou **2,08 vezes** a taxa geral da validação.

Com o limite padrão de probabilidade de **0,5**, precision e recall foram zero: nenhuma contratação foi corretamente classificada como positiva nesse limite. Antes de interpretar esse resultado, verificaremos quantas previsões positivas foram feitas e qual foi a maior probabilidade prevista. Para a decisão de quais clientes contatar, a qualidade da ordenação pode ser mais útil que esse limite fixo.


In [ ]:
print("Probabilidade máxima:", prob_validacao.max().round(3))
print("Previsões positivas com corte 0,5:", prev_validacao.sum())

contratacoes_top = int(y_validacao.iloc[indices_top].sum())
contratacoes_total = int(y_validacao.sum())

print("Contratações no top 10%:", contratacoes_top)
print("Contratações em toda a validação:", contratacoes_total)
print(f"Recall do top 10%: {contratacoes_top / contratacoes_total:.1%}")

In [ ]:
avaliacao = pd.DataFrame({
    "prob_prevista": prob_validacao,
    "contratou": y_validacao.to_numpy(),
})

print(f"Probabilidade média prevista: {avaliacao['prob_prevista'].mean():.1%}")
print(f"Contratação observada:       {avaliacao['contratou'].mean():.1%}")

avaliacao["faixa"] = pd.qcut(
    avaliacao["prob_prevista"].rank(method="first"),
    q=10,
    labels=range(1, 11)
)

display(
    avaliacao.groupby("faixa", observed=True)
    .agg(
        clientes=("contratou", "size"),
        probabilidade_media=("prob_prevista", "mean"),
        contratacao_observada=("contratou", "mean"),
    )
    .round(3)
)

### Ordenação e calibração das previsões

Na validação, a **probabilidade média prevista foi de 5,0%**, enquanto a **taxa observada de contratação foi de 11,1%**. O modelo, portanto, subestimou a frequência de contratações nesse período. Na faixa dos 10% de clientes com maior pontuação, a probabilidade média prevista foi de **8,3%**, mas **23,2%** contrataram.

Apesar dessa subestimação, o modelo apresentou alguma capacidade de **ordenar** clientes: a taxa de contratação foi de **6,4% na faixa de menor pontuação** e **23,2% na de maior pontuação**. A seleção dos 10% mais bem classificados reuniu **190 das 912 contratações (20,8%)**.

A diferença entre treino e validação pode contribuir para a subestimação, mas estes resultados não permitem atribuir a ela toda a causa. No próximo experimento, compararemos outro modelo usando a mesma divisão temporal e as mesmas variáveis, avaliando separadamente sua capacidade de ordenação e a calibração das probabilidades.


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

modelo_floresta = Pipeline([
    ("preprocessamento", clone(preprocessamento)),
    ("modelo", RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    )),
])

modelo_floresta.fit(X_treino, y_treino)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

def avaliar_validacao(nome, probabilidades):
    k = int(np.ceil(len(y_validacao) * 0.10))
    indices_top = np.argsort(probabilidades)[-k:]
    contratos_top = int(y_validacao.iloc[indices_top].sum())

    return {
        "modelo": nome,
        "average_precision": average_precision_score(y_validacao, probabilidades),
        "roc_auc": roc_auc_score(y_validacao, probabilidades),
        "prob_media": np.mean(probabilidades),
        "taxa_top_10": contratos_top / k,
        "contratos_top_10": contratos_top,
        "recall_top_10": contratos_top / y_validacao.sum(),
    }

prob_floresta = modelo_floresta.predict_proba(X_validacao)[:, 1]

resultados = pd.DataFrame([
    avaliar_validacao("Regressão logística", prob_validacao),
    avaliar_validacao("Random forest", prob_floresta),
]).set_index("modelo")

display(resultados.round(3))

In [ ]:
diagnostico = X_validacao[["poutcome"]].copy()
diagnostico["contratou"] = y_validacao.to_numpy()
diagnostico["prob_prevista"] = prob_validacao

display(
    diagnostico.groupby("poutcome")
    .agg(
        clientes=("contratou", "size"),
        taxa_observada=("contratou", "mean"),
        prob_media_prevista=("prob_prevista", "mean"),
    )
    .round(3)
)

A regressão logística subestimou a taxa de contratação em todos os grupos de resultado da campanha anterior. A random forest teve desempenho inferior na validação; por isso, manteremos a regressão logística e testaremos a inclusão de variáveis de contexto conhecidas antes do contato.

## 4. Variáveis de contexto e escolha do modelo

Adicionamos `month`, `day_of_week` e `contact`, assumindo que o mês, o dia e o canal planejado já são conhecidos no momento de preparar a lista de ligações. Caso o canal só seja registrado após o contato, ele não deve ser usado em um sistema prospectivo.

In [ ]:
colunas_contexto = colunas + ["month", "day_of_week", "contact"]

X_contexto = df[colunas_contexto]

X_contexto_treino = X_contexto.iloc[:fim_treino]
X_contexto_validacao = X_contexto.iloc[fim_treino:fim_validacao]

categoricas_contexto = (
    X_contexto_treino.select_dtypes(include="object").columns.tolist()
)
numericas_contexto = (
    X_contexto_treino.select_dtypes(exclude="object").columns.tolist()
)

In [ ]:
preprocessamento_contexto = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), categoricas_contexto),
    ("numericas", StandardScaler(), numericas_contexto),
])

modelo_contexto = Pipeline([
    ("preprocessamento", preprocessamento_contexto),
    ("modelo", LogisticRegression(max_iter=1000)),
])

modelo_contexto.fit(X_contexto_treino, y_treino)

prob_contexto = modelo_contexto.predict_proba(
    X_contexto_validacao
)[:, 1]

In [ ]:
comparacao = pd.DataFrame([
    avaliar_validacao("Logística inicial", prob_validacao),
    avaliar_validacao("Logística + contexto", prob_contexto),
]).set_index("modelo")

display(comparacao.round(3))

Na validação, a regressão logística com contexto elevou a *average precision* de **0,183 para 0,197** e as contratações no top 10% de **190 para 195**. A melhora foi modesta, mas consistente nas métricas de ordenação. O modelo foi escolhido usando somente a validação.

## 5. Avaliação final no período de teste

In [ ]:
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score

modelo_final = clone(modelo_contexto)

modelo_final.fit(
    X_contexto.iloc[:fim_validacao],
    y.iloc[:fim_validacao]
)

prob_teste = modelo_final.predict_proba(
    X_contexto.iloc[fim_validacao:]
)[:, 1]

k_teste = int(np.ceil(len(y_teste) * 0.10))
indices_top_teste = np.argsort(prob_teste)[-k_teste:]

contratos_top_teste = int(y_teste.iloc[indices_top_teste].sum())
contratos_totais_teste = int(y_teste.sum())
taxa_top_teste = contratos_top_teste / k_teste
taxa_geral_teste = y_teste.mean()

print(f"Taxa observada no teste: {taxa_geral_teste:.1%}")
print(f"Probabilidade média prevista: {prob_teste.mean():.1%}")
print(f"Average precision: {average_precision_score(y_teste, prob_teste):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_teste, prob_teste):.3f}")
print(f"Contratações no top 10%: {contratos_top_teste} de {contratos_totais_teste}")
print(f"Taxa de contratação no top 10%: {taxa_top_teste:.1%}")
print(f"Lift do top 10%: {taxa_top_teste / taxa_geral_teste:.2f}x")

### Resultado final

No conjunto de teste, a regressão logística com variáveis de contexto obteve **ROC-AUC de 0,696**. Entre os **10% de clientes com maior pontuação**, **423 dos 824** contrataram, uma taxa de **51,3%**, frente a **30,8%** no teste completo. Isso indica que o modelo é útil para **priorizar contatos**.

As probabilidades, porém, permaneceram subestimadas: a previsão média foi de **17,4%**, enquanto a contratação observada foi de **30,8%**. Assim, a ordenação dos clientes foi mais útil neste experimento do que interpretar cada pontuação como uma probabilidade precisa. A mudança nas taxas de contratação entre treino, validação e teste é uma limitação importante dos resultados.


## 6. Interpretação e diagnóstico exploratório

In [ ]:
from sklearn.inspection import permutation_importance

importancia = permutation_importance(
    modelo_contexto,
    X_contexto_validacao,
    y_validacao,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

tabela_importancia = (
    pd.DataFrame({
        "variavel": X_contexto_validacao.columns,
        "queda_media_ap": importancia.importances_mean,
        "desvio": importancia.importances_std,
    })
    .sort_values("queda_media_ap", ascending=False)
)

display(tabela_importancia.round(4))

### Interpretação das variáveis

Na validação, `month` foi a variável cuja permutação mais reduziu a *average precision* (queda média de **0,055**). Em seguida vieram as variáveis ligadas ao histórico das campanhas: `previous` (**0,039**), `pdays_informado` (**0,037**) e `poutcome` (**0,024**). Isso indica que o modelo utiliza principalmente informações sobre o período e os contatos anteriores para ordenar os clientes.

A importância de `month` exige cautela: como a taxa de contratação mudou ao longo dos registros, o mês pode estar representando condições específicas das campanhas, e não apenas sazonalidade. Já `education` e `day_of_week` apresentaram importâncias negativas nesta análise, mas isso, isoladamente, não demonstra que removê-las melhoraria o modelo.

A importância por permutação mede a contribuição das variáveis **para este modelo, nesta validação**. Ela não indica que uma variável cause contratações, e pode ser afetada pela informação compartilhada entre colunas como `previous`, `pdays_informado` e `poutcome`.


In [ ]:
meses = [
    "jan", "feb", "mar", "apr", "may", "jun",
    "jul", "aug", "sep", "oct", "nov", "dec"
]

analise_mes = df.iloc[:fim_validacao][["month", "y"]].copy()
analise_mes["periodo"] = (
    ["treino"] * fim_treino
    + ["validacao"] * (fim_validacao - fim_treino)
)
analise_mes["contratou"] = analise_mes["y"].eq("yes").astype(int)

resumo_mes = (
    analise_mes
    .groupby(["month", "periodo"])
    .agg(
        clientes=("contratou", "size"),
        taxa_contratacao=("contratou", "mean"),
    )
    .unstack("periodo")
    .reindex(meses)
)

display(resumo_mes.round(3))


In [ ]:
colunas_sem_mes = [c for c in colunas_contexto if c != "month"]

X_sem_mes = df[colunas_sem_mes]
X_sem_mes_treino = X_sem_mes.iloc[:fim_treino]
X_sem_mes_validacao = X_sem_mes.iloc[fim_treino:fim_validacao]

categoricas_sem_mes = (
    X_sem_mes_treino.select_dtypes(include="object").columns.tolist()
)
numericas_sem_mes = (
    X_sem_mes_treino.select_dtypes(exclude="object").columns.tolist()
)

preprocessamento_sem_mes = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), categoricas_sem_mes),
    ("numericas", StandardScaler(), numericas_sem_mes),
])

modelo_sem_mes = Pipeline([
    ("preprocessamento", preprocessamento_sem_mes),
    ("modelo", LogisticRegression(max_iter=1000)),
])

modelo_sem_mes.fit(X_sem_mes_treino, y_treino)

prob_sem_mes = modelo_sem_mes.predict_proba(
    X_sem_mes_validacao
)[:, 1]

comparacao_mes = pd.DataFrame([
    avaliar_validacao("Com mês", prob_contexto),
    avaliar_validacao("Sem mês", prob_sem_mes),
]).set_index("modelo")

display(comparacao_mes.round(3))

### Análise exploratória: contribuição de `month`

Para investigar a importância de `month`, retreinamos a regressão logística sem essa variável, mantendo os demais dados e a divisão temporal. Na validação, a *average precision* caiu de **0,197 para 0,158**, e a ROC-AUC de **0,643 para 0,576**. Entre os 10% de clientes com maior pontuação, o modelo sem `month` encontrou **157 contratações**, contra **195** no modelo com a variável.

Assim, `month` contribuiu para a ordenação dos clientes **neste período de validação**. Isso não demonstra uma sazonalidade estável: alguns meses aparecem apenas no treino ou apenas na validação, e a taxa de contratação mudou ao longo do tempo. Como esta comparação foi feita após a avaliação final no teste, ela é apresentada como **diagnóstico exploratório** e não altera o resultado final do projeto.


## Conclusões e limites

- A regressão logística com contexto ordenou os clientes melhor que uma seleção aleatória no período final: **423 das 2.540 contratações** estavam entre os **824 registros (10%)** com maior pontuação. A taxa nesse grupo foi **51,3%**, ante **30,8%** no conjunto de teste.
- A probabilidade média prevista no teste foi **17,4%**, abaixo dos **30,8% observados**. O modelo serve melhor para **ordenação** do que como estimativa calibrada da probabilidade de cada cliente.
- A taxa de contratação subiu de **4,8% no treino** para **11,1% na validação** e **30,8% no teste**. A mudança temporal limita a extrapolação. `month` contribuiu na validação, mas não foi demonstrada sazonalidade estável.
- A taxa e a composição do teste foram examinadas antes do treino. Assim, a avaliação final não é completamente cega, embora a escolha entre modelos tenha usado apenas a validação.
- A comparação sem `month` foi feita **após** a consulta ao teste e é apenas diagnóstica; não substitui a escolha nem o resultado final.
- O conjunto histórico corresponde a campanhas em Portugal entre 2008 e 2010. Seu desempenho não demonstra que o modelo funcionaria em outra época, instituição ou população. A interpretação por permutação mede dependência do modelo na validação, não causalidade.

**Próxima investigação possível:** testar a estabilidade em novas campanhas com datas completas e verificar a calibração antes de usar probabilidades para decisões de custo e retorno.